In [1]:
import asyncio
import os

import requests
from dotenv import load_dotenv

from agents import Agent, ModelSettings, Runner, function_tool, trace

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print("user set:", bool(pushover_user))
print("token set:", bool(pushover_token))

user set: True
token set: True


In [3]:
def push(message):
    print(f"Push: {message[:100]}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message[:1000]}
    response = requests.post(pushover_url, data=payload, timeout=10)
    print("Status:", response.status_code)

In [5]:
def send_message(subject, body):
    push(f"Subject: {subject}\n\n{body}")

In [6]:
send_message("Test subject", "Communications are a go!")

Push: Subject: Test subject

Communications are a go!
Status: 200


## three sales agents with different writing styles

In [7]:
intro = """
You are a sales agent working for CloudWatchly,
a company that provides a SaaS tool for monitoring cloud costs and flagging waste, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [9]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)

In [10]:
result = await Runner.run(sales_agent1, "Write a cold sales email")
print(result.final_output)

Subject: Reducing cloud waste without adding workload

Hi [First Name],

I’m reaching out from CloudWatchly. We help finance and engineering teams identify cloud cost waste, surface anomalies early, and prioritize the savings that matter most.

What sets us apart is that CloudWatchly uses AI to continuously monitor usage patterns and flag waste automatically, so your team spends less time hunting for inefficiencies and more time acting on them.

If cloud spend is under pressure, I’d welcome the chance to show you how we help companies regain control without adding operational burden.

Would you be open to a brief conversation next week?

Best,  
[Your Name]  
CloudWatchly


## run all three agents at the same time

In [11]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Reducing cloud waste without adding overhead

Hi [First Name],

I’m reaching out from CloudWatchly. We help teams monitor cloud spend, identify waste, and surface savings opportunities using AI—without adding more manual work to engineering or finance.

Most organizations know cloud costs are rising, but the harder problem is finding where the waste is hiding. CloudWatchly continuously flags idle resources, overprovisioning, and unusual spend patterns so your team can act quickly and with confidence.

If reducing cloud waste is a priority this quarter, I’d welcome a brief conversation to see whether we could be useful.

Best,  
[Your Name]  
CloudWatchly


Subject: Your cloud bill called. It has questions.

Hi [First Name],

I’ll keep this short, unlike most cloud invoices.

CloudWatchly helps teams spot cloud waste before it turns into a “how is this bill even possible?” moment. We monitor spend, flag inefficiencies, and use AI to surface the stuff that’s quietly burning budg

In [12]:
import time

start = time.time()
await Runner.run(sales_agent1, message)
await Runner.run(sales_agent2, message)
await Runner.run(sales_agent3, message)
print("One after another:", round(time.time() - start, 1), "seconds")

One after another: 7.7 seconds


In [13]:
start = time.time()
await asyncio.gather(
    Runner.run(sales_agent1, message),
    Runner.run(sales_agent2, message),
    Runner.run(sales_agent3, message),
)
print("Together:", round(time.time() - start, 1), "seconds")

Together: 3.7 seconds


In [14]:
## a picker agent chooses the best email (orchestration by code)
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales Picker", instructions=decision, model=MODEL_NAME)

In [15]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Cut cloud waste without adding overhead

Hi {{FirstName}},

Cloud spend often creeps up quietly — idle resources, overprovisioned instances, forgotten services.

CloudWatchly helps teams spot waste fast with AI-driven alerts and cost monitoring, so you can reduce spend without manual digging.

Worth a quick 15-minute chat to see if there’s an opportunity in your environment?

Best,  
{{YourName}}
